In [9]:
import os
import re
import copy
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# necessary for extending the vocabulary
import pickle
# necessary for importing the model
import sys
sys.path.append("nanoGPT")
from model import GPTConfig, GPT

# Load model S from task 1
BASE = "nanoGPT/data/shakespeare_char"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

block_size = 256
batch_size = 64
learning_rate = 5e-5
max_iters = 2000
eval_iters = 100

# load checkpoint of the model
ckpt = torch.load("nanoGPT/out-shakespeare-S-split100/ckpt.pt", map_location=DEVICE)
model = GPT(GPTConfig(**ckpt["model_args"]))
state_dict = ckpt["model"]
# remove _origin_mod because our model has been saved with torch.compile()
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

# extend vocabulary to fit our specially desigend tokens
# here we need pickle
meta_path = os.path.join(BASE, "meta.pkl")
with open(meta_path, "rb") as f:
    meta = pickle.load(f)

stoi = meta["stoi"]
itos = meta["itos"]

# Add special tokens
special_text = "[SPEAKER][ANSWER][END][CLASSIFY]"
for ch in special_text:
    if ch not in stoi:
        stoi[ch] = len(stoi)
        itos[len(itos)] = ch

vocab_size = len(stoi)

# extend embedding to fit the new tokens needed for our task "[", "]"
old_vocab_size = model.config.vocab_size
new_vocab_size = len(stoi)

if new_vocab_size > old_vocab_size:
    old_emb = model.transformer.wte.weight.data
    new_emb = torch.nn.Embedding(new_vocab_size, model.config.n_embd)
    new_emb.weight.data[:old_vocab_size] = old_emb
    model.transformer.wte = new_emb

    old_lm = model.lm_head.weight.data
    new_lm = torch.nn.Linear(model.config.n_embd, new_vocab_size, bias=False)
    new_lm.weight.data[:old_vocab_size] = old_lm
    model.lm_head = new_lm

    model.config.vocab_size = new_vocab_size

model.to(DEVICE)
model.eval()

# Define functions for encoding and decoding
def encode(text):
    return [stoi[c] for c in text]

def decode(indices):
    return "".join([itos[i] for i in indices])

# Define the batch sampler
def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.tensor(data[i:i+block_size]) for i in ix])
    y = torch.stack([torch.tensor(data[i+1:i+block_size+1]) for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

number of parameters: 0.80M


In [2]:
# Build dataset for speaker classification

with open(f"{BASE}/input.txt", "r") as f:
    lines = f.readlines()

speaker_pattern = re.compile(r"^([A-Z][A-Z ]+)[\.:]")

data = []
current_speaker = None

for line in lines:
    line = line.strip()
    match = speaker_pattern.match(line)
    if match:
        current_speaker = match.group(1)
    elif current_speaker and line:
        data.append((line, current_speaker))

speaker_counts = Counter([s for _, s in data])
top_10 = [s for s, _ in speaker_counts.most_common(10)]

filtered = [(l, s) for l, s in data if s in top_10]

taskA_examples = [
    f"[SPEAKER] {l} [ANSWER] {s} [END]"
    for l, s in filtered
]

# Split into train/test
random.shuffle(taskA_examples)

taskA_train = taskA_examples[:500]
taskA_test = taskA_examples[500:600]

# Save as text files
with open(f"{BASE}/sft_speaker_train.txt", "w") as f:
    for ex in taskA_train:
        f.write(ex + "\n")

with open(f"{BASE}/sft_speaker_test.txt", "w") as f:
    for ex in taskA_test:
        f.write(ex + "\n")

In [3]:
# Build dataset for verse/prose classification

def is_verse(block):
    lines = block.split("\n")
    avg_len = sum(len(l) for l in lines) / max(len(lines), 1)
    return avg_len < 60 and len(lines) >= 3
    
blocks = open(f"{BASE}/input.txt").read().split("\n\n")

taskB_examples = []

for b in blocks:
    if len(b.strip()) < 50:
        continue
    label = "VERSE" if is_verse(b) else "PROSE"
    text = f"[CLASSIFY] {b} [ANSWER] {label} [END]"
    taskB_examples.append(text)

# Split into train/test
random.shuffle(taskB_examples)
taskB_train = taskB_examples[:500]
taskB_test  = taskB_examples[500:600]

# Convert datasets into training format
def build_array(text_list):
    text = "\n".join(text_list)
    return np.array(encode(text), dtype=np.int64)

A_train_data = build_array(taskA_train)
A_test_data  = build_array(taskA_test)

B_train_data = build_array(taskB_train)
B_test_data  = build_array(taskB_test)


In [4]:
# Training function

def fine_tune(base_model, train_data, title):

    model = copy.deepcopy(base_model)
    model.to(DEVICE)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    losses = []

    for step in range(max_iters):
        xb, yb = get_batch(train_data)

        logits, loss = model(xb, yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        losses.append(loss.item())

        if step % 200 == 0:
            print(f"{title} | step {step} | loss {loss.item():.4f}")

    plt.plot(losses)
    plt.title(title)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(f"{title}_loss.png")
    plt.close()

    return model, losses

In [5]:
# Evaluation

def extract_answer(text):
    if "[ANSWER]" in text:
        return text.split("[ANSWER]")[1].split("[END]")[0].strip()
    return ""

def compute_taskA_accuracy(model):

    model.eval()
    correct = 0

    for ex in taskA_test:
        line = ex.split("[SPEAKER]")[1].split("[ANSWER]")[0].strip()
        label = extract_answer(ex)

        prompt = f"[SPEAKER] {line} [ANSWER]"
        idx = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)

        out = model.generate(idx, max_new_tokens=20)
        pred_text = decode(out[0].tolist())
        pred = extract_answer(pred_text)

        if pred == label:
            correct += 1

    return correct / len(taskA_test)

def compute_taskB_accuracy(model):

    model.eval()
    correct = 0

    for ex in taskB_test:
        label = extract_answer(ex)
        prompt = ex.split("[ANSWER]")[0] + "[ANSWER]"
        idx = torch.tensor(encode(prompt)).unsqueeze(0).to(DEVICE)

        out = model.generate(idx, max_new_tokens=20)
        pred_text = decode(out[0].tolist())
        pred = extract_answer(pred_text)

        if pred == label:
            correct += 1

    return correct / len(taskB_test)

# Compute validation loss, appended .astype(...) because we need long
val_data = np.memmap(
    f"{BASE}/val.bin",
    dtype=np.uint16,
    mode="r"
).astype(np.int64)

def estimate_val_loss(model):

    model.eval()
    losses = []

    for _ in range(eval_iters):
        xb, yb = get_batch(val_data)
        with torch.no_grad():
            _, loss = model(xb, yb)
        losses.append(loss.item())

    return sum(losses) / len(losses)

In [6]:
# Run Experiments

print("Running Single Task A")
model_A, _ = fine_tune(model, A_train_data, "Single_Task_A")

print("Running Single Task B")
model_B, _ = fine_tune(model, B_train_data, "Single_Task_B")

print("Running Multi Task A+B")
# randomize training data for both CHANGE
#mixed = np.concatenate([A_train_data, B_train_data])
combined = taskA_train + taskB_train
random.shuffle(combined)
mixed = build_array(combined)
model_multi, _ = fine_tune(model, mixed, "Multi_Task")

acc_A_single = compute_taskA_accuracy(model_A)
acc_B_single = compute_taskB_accuracy(model_B)

acc_A_multi = compute_taskA_accuracy(model_multi)
acc_B_multi = compute_taskB_accuracy(model_multi)

val_pre  = estimate_val_loss(model)
val_A    = estimate_val_loss(model_A)
val_B    = estimate_val_loss(model_B)
val_multi = estimate_val_loss(model_multi)

Running Single Task A
Single_Task_A | step 0 | loss 3.6566
Single_Task_A | step 200 | loss 0.9896
Single_Task_A | step 400 | loss 0.9020
Single_Task_A | step 600 | loss 0.8482
Single_Task_A | step 800 | loss 0.8101
Single_Task_A | step 1000 | loss 0.8020
Single_Task_A | step 1200 | loss 0.7794
Single_Task_A | step 1400 | loss 0.7436
Single_Task_A | step 1600 | loss 0.7497
Single_Task_A | step 1800 | loss 0.7171
Running Single Task B
Single_Task_B | step 0 | loss 2.2321
Single_Task_B | step 200 | loss 1.3886
Single_Task_B | step 400 | loss 1.2998
Single_Task_B | step 600 | loss 1.3071
Single_Task_B | step 800 | loss 1.2772
Single_Task_B | step 1000 | loss 1.2435
Single_Task_B | step 1200 | loss 1.2401
Single_Task_B | step 1400 | loss 1.2570
Single_Task_B | step 1600 | loss 1.2654
Single_Task_B | step 1800 | loss 1.2211
Running Multi Task A+B
Multi_Task | step 0 | loss 2.7634
Multi_Task | step 200 | loss 1.2938
Multi_Task | step 400 | loss 1.2912
Multi_Task | step 600 | loss 1.2469
Multi

In [7]:
# Print Results
results = pd.DataFrame({
    "Setup": ["Random", "Single A", "Single B", "Multi"],
    "Pretrained (No SFT)": [None, None, None, val_pre],
    "Task A Acc": [0.10, acc_A_single, None, acc_A_multi],
    "Task B Acc": [0.50, None, acc_B_single, acc_B_multi],
    "Val Loss": [None, val_A, val_B, val_multi]
})

results.to_csv("sft_results.csv", index=False)
print(results)

      Setup  Pretrained (No SFT)  Task A Acc  Task B Acc  Val Loss
0    Random                  NaN        0.10        0.50       NaN
1  Single A                  NaN        0.07         NaN  2.285536
2  Single B                  NaN         NaN        0.62  1.618787
3     Multi             1.584539        0.07        0.73  1.617823


In [8]:
# Generate samples

def generate_unconditional(model):
    idx = torch.zeros((1,1), dtype=torch.long).to(DEVICE)
    out = model.generate(idx, max_new_tokens=300)
    return decode(out[0].tolist())

with open("generated_samples.txt", "w") as f:
    f.write("=== PRETRAINED ===\n")
    f.write(generate_unconditional(model))
    f.write("\n\n=== SINGLE A ===\n")
    f.write(generate_unconditional(model_A))
    f.write("\n\n=== SINGLE B ===\n")
    f.write(generate_unconditional(model_B))
    f.write("\n\n=== MULTI ===\n")
    f.write(generate_unconditional(model_multi))